# 08 Geometric Differentiation

This standalone notebook treats Chapter 8 as one executable lab. The chapter is where geometric algebra starts behaving less like a catalog of products and more like a language for local change: a rotor is no longer only an operator, but also a path whose infinitesimal step can be measured; a vector variable is no longer only a point in a vector space, but a direction along which functions can be probed; a multivector function is no longer exotic, because every grade can be perturbed and checked with the same product rules.

The source heading audit for the requested printed-page span 213-244 is saved in `D:/Geometry/artifacts/chapter-08/headings.json`. In this PDF scan, the Chapter 8 extractable text begins on printed page 213 and the final extractable Chapter 8 page is printed page 241. The next extractable chapter text begins with Chapter 9 on printed page 245, so pages 242-244 appear to be non-content or non-extractable transition pages in this scan. The notebook follows the verified headings but uses original exposition, original examples, and executable checks rather than paraphrasing the source.

The recurring idea is that differentiating in geometric algebra means choosing the kind of variation first. A scalar parameter gives an ordinary one-dimensional derivative. A vector direction gives a directional derivative. Summing directional derivatives against reciprocal basis vectors gives the vector derivative. Perturbing every blade coordinate gives multivector differentiation. The syntax changes only a little, but the geometric meaning changes a lot: each derivative remembers whether the variation was a turn, a slide along a curve, a radial change, a tangent change, or a full algebraic perturbation.


## Lab Map

Work through the cells in order. The first half builds the calculus of transformations: small orthogonal changes, commutator products, scalar-parametric derivatives, and the special role of rotor-generated motion. The second half changes the variable being differentiated: first a vector in a chosen direction, then the full vector derivative, and finally a multivector line through the algebra.

Covered themes:

- orthogonal-transform changes as first-order commutator motion
- the commutator product and the Jacobi residual
- parametric differentiation of multivector-valued functions
- scalar differentiation of rotor transformations
- directional differentiation and the derivative of vector inversion
- the rotating mirror calculation as a local rotor
- vector differentiation as a basis-summed directional derivative
- multivector differentiation by perturbing blade coordinates
- spherical projection and its tangent-space derivative

A useful reading habit for this notebook is to keep three representations in view. The geometric object is the thing we mean. The multivector expression is the coordinate-free handle we use to compute with it. The numerical array or plot is only an audit trail. When those three agree across several perturbation sizes, the calculation has earned trust. When they disagree, the bug is usually a grade selection, a sign convention, or a left-versus-right product choice.


In [ ]:
from pathlib import Path
import json
import math
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "chapter08_differentiation.py").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.artifacts import display_artifact, save_json, save_matplotlib
from utils.chapter08_differentiation import (
    E2,
    E3,
    basis_blade_table,
    coefficient_vector,
    commutator,
    coords_to_vector,
    directional_derivative,
    first_order_rotor_change,
    jacobi_residual,
    mirror_tilt_derivative,
    multivector_from_coefficients,
    reflect_vector_in_plane,
    rotor_transform,
    scalar_central_difference,
    scalar_norm_squared,
    spherical_projection,
    spherical_projection_derivative,
    vector_derivative,
    vector_inverse,
    vector_inverse_derivative,
    vector_to_coords,
)

np.set_printoptions(precision=5, suppress=True)
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
ARTIFACT_TOPIC = "chapter-08"
CHAPTER_DIR = (
    PROJECT_ROOT
    / "Geometric-Algebra-for-Computer-Science"
    / "part-01-geometric-algebra"
    / "chapter-08-geometric-differentiation"
)
NOTEBOOK_PATH = CHAPTER_DIR / "08-geometric-differentiation.ipynb"

e1, e2, e3 = E3.basis()
f1, f2 = E2.basis()
I2 = f1.wedge(f2)
B12 = e1.wedge(e2)
B23 = e2.wedge(e3)
B31 = e3.wedge(e1)

print(PROJECT_ROOT)


In [ ]:
headings_path = ARTIFACT_ROOT / ARTIFACT_TOPIC / "headings.json"
heading_audit = json.loads(headings_path.read_text(encoding="utf-8"))
heading_rows = pd.DataFrame(heading_audit["headings"])
print("requested printed pages:", heading_audit["requested_printed_pages"])
print("extractable Chapter 8 pages:", heading_audit["verified_extractable_chapter_pages"])
display(heading_rows)


## Orthogonal-Transform Changes

An orthogonal transformation preserves the metric. In geometric algebra that usually means a versor action, and for rotations it means a rotor sandwich. Chapter 8 asks what happens when the operator itself changes only a little. If the plane bivector `B` generates a small rotor, the transformed object is not best understood as a new coordinate formula; it is the old object plus the commutator with the generator. That is a compact statement of local rotation.

For a rotor `R(eps) = exp(-eps B/2)`, the sandwich `R X reverse(R)` has the first-order approximation `X + eps (X x B)`, where `x` denotes the commutator product. The equality is not exact for a finite angle because a finite turn contains second-order and higher terms, but the error should scale quadratically as the angle shrinks. That scaling is the computational signature of a valid derivative.

This is the first place where the algebraic product earns its keep. The same expression can move a vector, a bivector, or a mixed multivector, and the code does not need a special coordinate formula for each grade. A small orthogonal change is therefore represented by a bivector generator rather than by a matrix of partial derivatives.


In [ ]:
X = 1.2 * e1 - 0.35 * e2 + 0.6 * e3
angles = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
rows = []
for eps in angles:
    exact = rotor_transform(X, B12, float(eps)).grade(1)
    approx = first_order_rotor_change(X, B12, float(eps)).grade(1)
    error = np.linalg.norm(coefficient_vector(exact - approx))
    rows.append(
        {
            "small_angle": float(eps),
            "exact": repr(exact),
            "first_order": repr(approx),
            "error_norm": float(error),
            "error_over_angle_squared": float(error / eps**2),
        }
    )

path = save_json(rows, ARTIFACT_TOPIC, "checks", "rotor-first-order.json", root=ARTIFACT_ROOT)
assert rows[-1]["error_norm"] < rows[0]["error_norm"]
print(f"wrote {path}")
pd.DataFrame(rows)


In [ ]:
grid = np.linspace(-1.5, 1.5, 9)
points = []
changes = []
for x in grid:
    for y in grid:
        vector = coords_to_vector([x, y, 0.0])
        delta = commutator(vector, B12).grade(1)
        points.append([x, y])
        changes.append(vector_to_coords(delta)[:2])
points = np.array(points)
changes = np.array(changes)

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(points[:, 0], points[:, 1], changes[:, 0], changes[:, 1], color="#1f6f8b", angles="xy")
ax.set_aspect("equal")
ax.set_title("Infinitesimal rotor field from x x (e1^e2)")
ax.set_xlabel("e1 coordinate")
ax.set_ylabel("e2 coordinate")
ax.axhline(0, color="0.85", linewidth=1)
ax.axvline(0, color="0.85", linewidth=1)
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
field_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "commutator-vector-field.png", root=ARTIFACT_ROOT)
plt.close(fig)
display_artifact(field_path, width=520)


## The Commutator Product

The commutator product is half the difference between the two possible geometric-product orders. That half-difference is exactly the part that notices noncommutation. With a bivector in one slot, the product acts like an infinitesimal rotation generator and preserves grade in the cases used here. For example, a vector commuted with a Euclidean plane bivector is another vector, while a bivector commuted with a bivector is another bivector.

The commutator is bilinear, but it is not associative. That nonassociativity is controlled rather than chaotic: the Jacobi sum vanishes. In practical terms, the Jacobi identity tells us that nested infinitesimal rotations have a consistent algebra. The small changes do not behave like ordinary scalar increments; they form a Lie algebra of bivectors. This is why a sequence of tiny orthogonal transformations can be integrated into a finite rotor.

The next cell checks three things numerically. First, it shows the grade behavior when the right argument is a bivector. Second, it records a deliberately nested commutator so the nonassociativity is visible. Third, it computes the Jacobi residual and verifies that it vanishes to the tolerance of the small symbolic algebra used in this course.


In [ ]:
objects = {
    "vector X": X,
    "bivector e2^e3": B23,
    "mixed X + e2^e3": X + B23,
}
rows = []
for name, value in objects.items():
    result = commutator(value, B12)
    rows.append(
        {
            "input": name,
            "input_grades": sorted(value.grades()),
            "commutator_with_B12": repr(result),
            "result_grades": sorted(result.grades()),
        }
    )

left_nested = commutator(commutator(B12, B12), B23)
right_nested = commutator(B12, commutator(B12, B23))
residual = jacobi_residual(B12, B12, B23)
assert not left_nested.almost_equal(right_nested)
assert np.linalg.norm(coefficient_vector(residual)) < 1e-9
rows.append(
    {
        "input": "nested bivectors",
        "input_grades": [2],
        "commutator_with_B12": f"left={left_nested}; right={right_nested}",
        "result_grades": sorted((left_nested - right_nested).grades()),
    }
)
rows.append(
    {
        "input": "Jacobi residual",
        "input_grades": [2],
        "commutator_with_B12": repr(residual),
        "result_grades": sorted(residual.grades()),
    }
)
pd.DataFrame(rows)


## Parametric Differentiation

A parametric derivative is the most familiar case: a scalar parameter changes, and a multivector-valued function changes with it. Geometric algebra does not replace the ordinary product rule; it gives the product rule more interesting objects to act on. A parameterized vector can be wedged with its velocity, a time-dependent plane can become a time-dependent bivector, and a rotor path can generate a changing sandwich.

The important discipline is to differentiate the construction, not only its coordinates. If `A(t) = r(t) ^ v(t)`, then its derivative is `r'(t) ^ v(t) + r(t) ^ v'(t)`. This looks like the ordinary product rule because the outer product is bilinear. The benefit is that the result is still a bivector: it is an oriented area-rate, not a list of unrelated component derivatives.

The example below uses a planar curve that is not a perfect circle. At one chosen parameter value it compares a central finite difference with the analytic product-rule result. The figure is saved so the artifact folder records the curve, the sampled point, the velocity direction, and the local oriented area element being differentiated.


In [ ]:
def r(t: float):
    return coords_to_vector([math.cos(t) + 0.25 * math.cos(2.0 * t), math.sin(t)], E2)


def r_dot(t: float):
    return coords_to_vector([-math.sin(t) - 0.5 * math.sin(2.0 * t), math.cos(t)], E2)


def r_ddot(t: float):
    return coords_to_vector([-math.cos(t) - math.cos(2.0 * t), -math.sin(t)], E2)


def swept_bivector(t: float):
    return r(t).wedge(r_dot(t))


t0 = 0.85
numeric = scalar_central_difference(swept_bivector, t0)
analytic = r_dot(t0).wedge(r_dot(t0)) + r(t0).wedge(r_ddot(t0))
assert numeric.almost_equal(analytic, tol=1e-7)

curve_t = np.linspace(0, 2 * np.pi, 300)
curve = np.array([vector_to_coords(r(float(t))) for t in curve_t])
point = vector_to_coords(r(t0))
velocity = vector_to_coords(r_dot(t0))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(curve[:, 0], curve[:, 1], color="#355c7d", linewidth=2)
ax.scatter([point[0]], [point[1]], color="#c44536", zorder=3)
ax.quiver(point[0], point[1], velocity[0], velocity[1], angles="xy", scale_units="xy", scale=3.0, color="#2a9d8f")
ax.fill(
    [0, point[0], point[0] + velocity[0] / 3.0, velocity[0] / 3.0],
    [0, point[1], point[1] + velocity[1] / 3.0, velocity[1] / 3.0],
    alpha=0.16,
    color="#c44536",
)
ax.set_aspect("equal")
ax.set_title("Parametric bivector A(t) = r(t)^r'(t)")
ax.set_xlabel("e1 coordinate")
ax.set_ylabel("e2 coordinate")
curve_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "parametric-bivector-rate.png", root=ARTIFACT_ROOT)
plt.close(fig)

save_json(
    {
        "t0": t0,
        "finite_difference": repr(numeric),
        "product_rule": repr(analytic),
        "figure": str(curve_path),
    },
    ARTIFACT_TOPIC,
    "checks",
    "parametric-product-rule.json",
    root=ARTIFACT_ROOT,
)

display_artifact(curve_path, width=560)
pd.DataFrame(
    [
        {"quantity": "finite difference", "value": repr(numeric)},
        {"quantity": "product rule", "value": repr(analytic)},
    ]
)


## Scalar Differentiation of Rotor Motion

Scalar differentiation becomes especially revealing when the scalar controls a rotor. Let a vector `X0` be transformed by a rotor whose angle is `theta(t)`. The derivative of the transformed vector is not merely the derivative of sine and cosine components; geometrically it is the current transformed vector commuted with the current angular-rate bivector.

This is the bridge between ordinary time derivatives and the later vector derivatives. The scalar parameter has only one direction in its domain, but the value of the function lives in the algebra. The derivative is therefore multivector-valued. If the rotor angle has rate `theta'(t)`, then the first-order change is governed by `theta'(t) B`, the bivector that says which plane is turning and how quickly.

The check below differentiates a rotor path numerically and compares it with the commutator formula at the same parameter value. The point is not that finite differences are elegant; it is that they make the local claim falsifiable. Once the finite-difference derivative agrees with the commutator expression, later sections can reuse the commutator as a geometric derivative rather than as a coordinate trick.


In [ ]:
theta = lambda t: 0.4 + 0.35 * t * t
theta_dot = lambda t: 0.7 * t
X0 = 1.2 * e1 + 0.4 * e2 + 0.6 * e3


def moving_vector(t: float):
    return rotor_transform(X0, B12, theta(t)).grade(1)


t0 = 1.2
numeric = scalar_central_difference(moving_vector, t0)
analytic = theta_dot(t0) * commutator(moving_vector(t0), B12)
error = float(np.linalg.norm(coefficient_vector(numeric - analytic)))
assert error < 1e-7
path = save_json(
    {
        "t0": t0,
        "theta": theta(t0),
        "theta_dot": theta_dot(t0),
        "finite_difference": repr(numeric),
        "commutator_formula": repr(analytic),
        "error_norm": error,
    },
    ARTIFACT_TOPIC,
    "checks",
    "scalar-rotor-derivative.json",
    root=ARTIFACT_ROOT,
)
print(f"wrote {path}")
pd.DataFrame(
    [
        {"quantity": "finite difference", "value": repr(numeric)},
        {"quantity": "commutator", "value": repr(analytic)},
        {"quantity": "error norm", "value": error},
    ]
)


## Directional Differentiation and Inversion

A directional derivative changes the input vector by a small amount in one chosen vector direction. The notation is often read as “differentiate at `x` in the direction `a`.” That is different from a scalar derivative because the domain itself is a vector space; many directions are available, and the answer depends on which one is chosen.

Vector inversion is a good stress test. The inverse of a nonzero Euclidean vector is `x / x^2`. If `x` changes by `a`, the derivative is not simply the derivative of a scalar reciprocal; the direction `a` is reflected and scaled through the current inverse. In compact geometric-product form, the full-space directional derivative is `-x^{-1} a x^{-1}`.

The plot below shows a small local picture of this derivative. The original vector and its inverse are both drawn from the origin. At the inverse point, the derivative arrow indicates how the inverse moves when the original vector starts moving in direction `a`. Notice the derivative can point in a direction that is not parallel to `a`; inversion couples radial and angular changes.


In [ ]:
x = coords_to_vector([1.2, 0.8], E2)
a = coords_to_vector([0.35, -0.2], E2)
finite = directional_derivative(vector_inverse, x, a)
analytic = vector_inverse_derivative(x, a).grade(1)
assert finite.almost_equal(analytic, tol=1e-7)

x_arr = vector_to_coords(x)
a_arr = vector_to_coords(a)
inv_arr = vector_to_coords(vector_inverse(x))
deriv_arr = vector_to_coords(analytic)
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.axhline(0, color="0.86", linewidth=1)
ax.axvline(0, color="0.86", linewidth=1)
ax.quiver(0, 0, x_arr[0], x_arr[1], angles="xy", scale_units="xy", scale=1, color="#355c7d", label="x")
ax.quiver(x_arr[0], x_arr[1], a_arr[0], a_arr[1], angles="xy", scale_units="xy", scale=1, color="#2a9d8f", label="a at x")
ax.quiver(0, 0, inv_arr[0], inv_arr[1], angles="xy", scale_units="xy", scale=1, color="#c44536", label="x^{-1}")
ax.quiver(inv_arr[0], inv_arr[1], deriv_arr[0], deriv_arr[1], angles="xy", scale_units="xy", scale=1, color="#6d597a", label="derivative")
ax.set_aspect("equal")
ax.set_xlim(-0.2, 1.8)
ax.set_ylim(-0.4, 1.35)
ax.set_title("Directional derivative of vector inversion")
ax.legend(loc="upper right")
inversion_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "inversion-directional-derivative.png", root=ARTIFACT_ROOT)
plt.close(fig)

save_json(
    {"x": repr(x), "direction": repr(a), "finite_difference": repr(finite), "analytic": repr(analytic), "figure": str(inversion_path)},
    ARTIFACT_TOPIC,
    "checks",
    "inversion-derivative.json",
    root=ARTIFACT_ROOT,
)
display_artifact(inversion_path, width=540)
pd.DataFrame(
    [
        {"quantity": "finite difference", "value": repr(finite)},
        {"quantity": "-x^{-1} a x^{-1}", "value": repr(analytic)},
    ]
)


## Rotating Mirror

A mirror can be described by its normal vector. Changing the magnitude of that normal does not change the mirror, but changing its direction tilts the mirror and changes the reflected vector. This makes the mirror problem a perfect directional-derivative example: the meaningful part of the normal change is the part that rotates the normal, not the part that merely rescales it.

For a vector reflection in a plane with normal `n`, the reflected vector is `-n x n^{-1}`. Perturb the normal by a small vector `a`, and the derivative of the reflected vector can be written as a commutator of the reflected vector with the bivector `2 n^{-1} ^ a`. That bivector is the local generator of the mirror's tilt. The derivative therefore says more than “the reflected coordinate changed”; it says that, to first order, the reflected result is itself undergoing a rotor-like motion.

The code compares the finite-difference derivative with that commutator expression. The figure shows the original mirror, a visibly tilted nearby mirror, the incoming vector, and the two reflected vectors. The visible tilt is larger than the derivative step so the geometry is legible; the numeric derivative still uses a tiny symmetric step.


In [ ]:
mirror_vector = coords_to_vector([1.05, 0.55], E2)
normal = f2
normal_change = 0.45 * f1 + 0.05 * f2


def reflection_for_normal(n):
    return reflect_vector_in_plane(mirror_vector, n)


finite = directional_derivative(reflection_for_normal, normal, normal_change)
analytic = mirror_tilt_derivative(mirror_vector, normal, normal_change)
assert finite.almost_equal(analytic, tol=1e-7)

normal_tilted = normal + 0.45 * normal_change
incoming = vector_to_coords(mirror_vector)
reflected = vector_to_coords(reflect_vector_in_plane(mirror_vector, normal))
reflected_tilted = vector_to_coords(reflect_vector_in_plane(mirror_vector, normal_tilted))


def mirror_line_points(n, length=1.6):
    nxy = vector_to_coords(n)
    direction = np.array([-nxy[1], nxy[0]])
    direction = direction / np.linalg.norm(direction)
    return np.vstack([-length * direction, length * direction])


fig, ax = plt.subplots(figsize=(6.5, 5.5))
line = mirror_line_points(normal)
tilted_line = mirror_line_points(normal_tilted)
ax.plot(line[:, 0], line[:, 1], color="0.25", linewidth=2, label="mirror")
ax.plot(tilted_line[:, 0], tilted_line[:, 1], color="#2a9d8f", linewidth=2, linestyle="--", label="tilted mirror")
ax.quiver(0, 0, incoming[0], incoming[1], angles="xy", scale_units="xy", scale=1, color="#355c7d", label="x")
ax.quiver(0, 0, reflected[0], reflected[1], angles="xy", scale_units="xy", scale=1, color="#c44536", label="reflected")
ax.quiver(0, 0, reflected_tilted[0], reflected_tilted[1], angles="xy", scale_units="xy", scale=1, color="#6d597a", label="tilted reflected")
ax.set_aspect("equal")
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.2, 1.2)
ax.set_title("Tilting a mirror changes the reflected vector")
ax.legend(loc="upper right")
mirror_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "rotating-mirror.png", root=ARTIFACT_ROOT)
plt.close(fig)

save_json(
    {"finite_difference": repr(finite), "commutator_formula": repr(analytic), "figure": str(mirror_path)},
    ARTIFACT_TOPIC,
    "checks",
    "rotating-mirror.json",
    root=ARTIFACT_ROOT,
)
display_artifact(mirror_path, width=560)
pd.DataFrame(
    [
        {"quantity": "finite difference", "value": repr(finite)},
        {"quantity": "commutator form", "value": repr(analytic)},
    ]
)


## Vector Differentiation

Directional derivatives answer one-direction questions. The vector derivative collects all basis directions into one operator. In an orthonormal Euclidean basis the implementation is direct: differentiate along each basis vector and multiply the result on the left by that basis vector, then sum. The result may have several grades because the geometric product is being used inside the operator.

Two simple checks anchor the idea. First, the vector derivative of the identity vector field in three dimensions is the scalar `3`, one contribution from each basis direction. Second, the vector derivative of the scalar function `x^2` is `2x`, just as the ordinary gradient of squared length would suggest. These are familiar answers, but here they emerge from the same product-valued operator that can also produce bivector and higher-grade parts for other functions.

This view helps interpret divergence and curl without switching notations. The scalar part of `partial_x F` records expansion-like behavior of a vector field. The bivector part records circulation in oriented coordinate planes. A single geometric derivative contains both, and grade selection recovers whichever classical piece a calculation needs.


In [ ]:
point = coords_to_vector([1.2, -0.4, 0.7], E3)
identity_derivative = vector_derivative(lambda y: y, point)
norm_derivative = vector_derivative(scalar_norm_squared, point)
expected_identity = E3.scalar(3)
expected_norm = 2.0 * point
assert identity_derivative.almost_equal(expected_identity, tol=1e-7)
assert norm_derivative.almost_equal(expected_norm, tol=1e-7)

swirl = lambda y: commutator(y, B12).grade(1)
swirl_derivative = vector_derivative(swirl, point)
rows = [
    {"function": "F(x) = x", "vector_derivative": repr(identity_derivative), "expected": repr(expected_identity), "grades": sorted(identity_derivative.grades())},
    {"function": "F(x) = x^2", "vector_derivative": repr(norm_derivative), "expected": repr(expected_norm), "grades": sorted(norm_derivative.grades())},
    {"function": "F(x) = x x (e1^e2)", "vector_derivative": repr(swirl_derivative), "expected": "pure bivector circulation", "grades": sorted(swirl_derivative.grades())},
]
path = save_json(rows, ARTIFACT_TOPIC, "checks", "vector-derivative-examples.json", root=ARTIFACT_ROOT)
print(f"wrote {path}")
pd.DataFrame(rows)


## Multivector Differentiation

Vector differentiation still assumes the input variable is a vector. Multivector differentiation removes that restriction. A multivector has scalar, vector, bivector, and higher-grade coordinates, so a full derivative can perturb any blade coordinate. Conceptually this is not a different kind of magic; it is the same directional idea applied to the vector space whose basis is the set of basis blades.

The example below keeps the function deliberately simple: `F(M) = M C`, where `C` is a fixed multivector. Along a multivector direction `A`, the derivative should be `A C` because the function is linear in `M`. The finite-difference check confirms this. Then the cell perturbs each basis blade separately and records the corresponding directional derivative. That table is a small coordinate atlas for the multivector derivative.

This full-algebra view matters because many geometric quantities are not pure vectors. A rotor is even-grade. An oriented plane is a bivector. A pseudoscalar may encode a local volume element. Differentiating only vector coordinates would miss the natural directions in which those objects change. Multivector differentiation lets the calculus follow the object instead of forcing the object into vector clothing.


In [ ]:
C = multivector_from_coefficients([1.0, 0.2, -0.1, 0.35, 0.4, -0.25, 0.15, 0.05], E3)
M = multivector_from_coefficients([0.5, 1.1, -0.3, 0.2, 0.7, -0.15, 0.05, 0.4], E3)
A = multivector_from_coefficients([0.0, -0.25, 0.4, 0.1, -0.2, 0.3, 0.05, -0.1], E3)
F = lambda Y: Y.gp(C)
finite = directional_derivative(F, M, A)
analytic = A.gp(C)
assert finite.almost_equal(analytic, tol=1e-7)

basis_rows = []
for row in basis_blade_table(E3):
    blade = E3.blade(row["mask"])
    derivative = directional_derivative(F, M, blade)
    basis_rows.append(
        {
            "blade": row["blade"],
            "grade": row["grade"],
            "directional_derivative": repr(derivative),
            "result_grades": sorted(derivative.grades()),
        }
    )
path = save_json(
    {"direction_A_finite_difference": repr(finite), "direction_A_analytic": repr(analytic), "basis_table": basis_rows},
    ARTIFACT_TOPIC,
    "checks",
    "multivector-directional-derivative.json",
    root=ARTIFACT_ROOT,
)
print(f"wrote {path}")
display(pd.DataFrame([{"quantity": "finite difference in A", "value": repr(finite)}, {"quantity": "A C", "value": repr(analytic)}]))
pd.DataFrame(basis_rows)


## Spherical Projection

The exercise on spherical projection asks for the directional derivative of `P(x) = x / ||x||`. The geometry is clean: projecting to the unit sphere removes radial information, so a purely radial change should not move the projected point to first order. Only the tangent component of the direction matters, scaled by the original radius.

In geometric algebra, the tangent component of a direction `a` relative to `x` can be written with the outer product: `(a ^ x) x^{-1}`. This rejects the part of `a` parallel to `x` and keeps the part tangent to the sphere through `x / ||x||`. Dividing by `||x||` gives the derivative of the normalized vector. The formula is compact, but the picture is often the best explanation: the derivative arrow lies in the tangent plane at the projected point.

The following cell checks the formula by finite differences in three dimensions and saves a wireframe sphere figure. The original vector reaches out to space, the projection lands on the unit sphere, and the derivative arrow sits tangent to that sphere. A dot-product assertion verifies tangency numerically by checking that the derivative is orthogonal to the projected unit vector.


In [ ]:
x3 = coords_to_vector([1.4, 0.8, 0.9], E3)
a3 = coords_to_vector([0.5, -0.7, 0.4], E3)
finite = directional_derivative(spherical_projection, x3, a3)
analytic = spherical_projection_derivative(x3, a3)
projected = spherical_projection(x3)
assert finite.almost_equal(analytic, tol=1e-7)
tangency = projected.scalar_product(analytic).scalar_value()
assert abs(tangency) < 1e-9

x_arr = vector_to_coords(x3)
p_arr = vector_to_coords(projected)
d_arr = vector_to_coords(analytic)
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
u = np.linspace(0, 2 * np.pi, 40)
v = np.linspace(0, np.pi, 20)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones_like(u), np.cos(v))
ax.plot_wireframe(xs, ys, zs, color="0.82", linewidth=0.6)
ax.quiver(0, 0, 0, x_arr[0], x_arr[1], x_arr[2], color="#355c7d", linewidth=2, arrow_length_ratio=0.08)
ax.scatter([p_arr[0]], [p_arr[1]], [p_arr[2]], color="#c44536", s=40)
ax.quiver(p_arr[0], p_arr[1], p_arr[2], d_arr[0], d_arr[1], d_arr[2], color="#2a9d8f", linewidth=2, arrow_length_ratio=0.2)
ax.set_title("Directional derivative of spherical projection")
ax.set_box_aspect((1, 1, 1))
ax.set_xlim(-1.2, 1.6)
ax.set_ylim(-1.2, 1.2)
ax.set_zlim(-1.2, 1.2)
projection_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "spherical-projection-derivative.png", root=ARTIFACT_ROOT)
plt.close(fig)

save_json(
    {
        "x": repr(x3),
        "direction": repr(a3),
        "finite_difference": repr(finite),
        "analytic": repr(analytic),
        "projected_dot_derivative": tangency,
        "figure": str(projection_path),
    },
    ARTIFACT_TOPIC,
    "checks",
    "spherical-projection.json",
    root=ARTIFACT_ROOT,
)
display_artifact(projection_path, width=560)
pd.DataFrame(
    [
        {"quantity": "finite difference", "value": repr(finite)},
        {"quantity": "tangent formula", "value": repr(analytic)},
        {"quantity": "P(x) dot derivative", "value": tangency},
    ]
)


## Sanity Checks

The notebook ends with a concrete audit. The checks are intentionally mundane: count the markdown words, count the code cells, verify the heading audit exists, and verify that the main figures and JSON outputs were written to disk. This keeps the deliverable honest. A notebook about differentiation should not only describe local change; it should also leave reproducible evidence that the stated formulas survived numerical perturbation tests.

A few limitations are worth naming. The finite differences use small real steps, so they check local formulas numerically rather than proving them symbolically. The helper algebra is an educational implementation for orthogonal metrics, not a production symbolic engine. The multivector derivative section illustrates the coordinate idea with a linear function rather than attempting the full theory of geometric calculus. Those constraints are deliberate: the goal is a reliable, inspectable lab that makes the chapter's main operators concrete before later chapters build larger models on top of them.


In [ ]:
expected_artifacts = [
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "headings.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "rotor-first-order.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "parametric-product-rule.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "scalar-rotor-derivative.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "inversion-derivative.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "rotating-mirror.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "vector-derivative-examples.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "multivector-directional-derivative.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "checks" / "spherical-projection.json",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "figures" / "commutator-vector-field.png",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "figures" / "parametric-bivector-rate.png",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "figures" / "inversion-directional-derivative.png",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "figures" / "rotating-mirror.png",
    ARTIFACT_ROOT / ARTIFACT_TOPIC / "figures" / "spherical-projection-derivative.png",
]
missing = [str(path) for path in expected_artifacts if not path.exists()]
assert not missing, missing

notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
markdown_words = 0
for cell in notebook["cells"]:
    if cell["cell_type"] == "markdown":
        markdown_words += len(re.findall(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?", "".join(cell["source"])))
code_cells = sum(1 for cell in notebook["cells"] if cell["cell_type"] == "code")
assert markdown_words >= 2000
assert code_cells >= 8

sanity = {
    "notebook": str(NOTEBOOK_PATH),
    "markdown_words": markdown_words,
    "code_cells": code_cells,
    "artifact_count_checked": len(expected_artifacts),
    "page_span_requested": [213, 244],
    "extractable_chapter_pages": heading_audit["verified_extractable_chapter_pages"],
}
sanity_path = save_json(sanity, ARTIFACT_TOPIC, "checks", "sanity-checks.json", root=ARTIFACT_ROOT)
print(f"wrote {sanity_path}")
sanity
